# CoralGuard-X V2 FINAL — Teacher → Student Hybrid Coral Pipeline

**LOCAL Windows / NVIDIA GPU**

Workspace:

```text
D:\Coral_Local_Workspace
```

## Final architecture

```text
                         OFFLINE TRAINING
═══════════════════════════════════════════════════════════════

Existing YOLO26m best detector
        │
        ├──────── compatible backbone/neck/box weights ───────┐
        │                                                     │
GT coral boxes                                                │
        ↓                                                     │
CoralSCOP Teacher                                             │
        ↓                                                     │
Pseudo instance masks                                        │
        ↓                                                     │
Automatic mask-quality filter                                │
        ↓                                                     │
YOLO26m-SEG STUDENT  ◄────────────────────────────────────────┘
        ↓
FINAL coral instance segmenter
(Box + Mask + Confidence)

Teacher masks + 10% context
        ↓
Dual-View EfficientNetV2-S
        ↓
6 known coral labels + NON_CORAL
        ↓
Embedding OOD gate
        ↓
Known species / Other-Unknown Coral / Non-Coral


                           DEPLOYMENT
═══════════════════════════════════════════════════════════════

Image / Video
        ↓
YOLO26m-SEG Student
        ↓
Box + instance mask
        ↓
Mask-quality / overlap gate
        │
        ├── good mask ─────────────────────┐
        │                                  │
        └── difficult mask → CoralSCOP ────┤
                                           ↓
                              Final precise instance mask
                                           ↓
                              Masked ROI + 10% Context
                                           ↓
                              Dual-View EfficientNetV2-S
                                           ↓
                              Known / Unknown / Non-Coral
                                           ↓
                              SAM2.1 temporal mask memory
                                           ↓
                              Stable ID + mask + species
```

## Important

- This implements the **Teacher → Student idea for YOLO**, not just CoralSCOP refinement at inference.
- The segmentation student is `YOLO26m-seg`.
- CoralSCOP generates pseudo masks from the existing GT boxes.
- The existing coral YOLO detector transfers all compatible weights into the segmentation student.
- The held-out detector **test** split is never used for training or threshold tuning.
- Because there are no human instance masks yet, segmentation validation measures agreement with **CoralSCOP pseudo masks**, not independent human-mask ground truth.
- The current dataset does not contain explicit `Dead Coral` / `Algae` labels, so those names are not invented.


In [ ]:
# CELL 1 — LOCAL ENVIRONMENT

from pathlib import Path
from collections import defaultdict
import os
import sys
import gc
import csv
import json
import math
import random
import shutil
import time
import zipfile
import subprocess
import importlib.util

import numpy as np
import pandas as pd
import cv2
from PIL import Image

import torch
import torch.nn as nn

from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torchvision import transforms
from torchvision.transforms import InterpolationMode

import timm

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix,
)

from scipy.optimize import linear_sum_assignment

from ultralytics import YOLO
import ultralytics


SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)


WORKSPACE = Path(r"D:\Coral_Local_Workspace")
VENV_PYTHON = WORKSPACE / ".venv" / "Scripts" / "python.exe"

if not WORKSPACE.exists():
    raise FileNotFoundError(WORKSPACE)

if not VENV_PYTHON.exists():
    raise FileNotFoundError(VENV_PYTHON)

if not torch.cuda.is_available():
    raise RuntimeError(
        "CUDA is unavailable. In VS Code select:\n"
        r"D:\Coral_Local_Workspace\.venv\Scripts\python.exe"
    )


DEVICE = torch.device("cuda")
GPU_NAME = torch.cuda.get_device_name(0)
VRAM_GB = torch.cuda.get_device_properties(0).total_memory / (1024 ** 3)


# RTX A6000 48GB defaults.
if VRAM_GB >= 38:
    SEG_BATCH = 8
    CLS_BATCH = 64
elif VRAM_GB >= 22:
    SEG_BATCH = 6
    CLS_BATCH = 32
elif VRAM_GB >= 14:
    SEG_BATCH = 4
    CLS_BATCH = 16
else:
    SEG_BATCH = 2
    CLS_BATCH = 8


print("=" * 110)
print("CORALGUARD-X V2 LOCAL ENVIRONMENT")
print("=" * 110)
print("Python       :", sys.executable)
print("Workspace    :", WORKSPACE)
print("GPU          :", GPU_NAME)
print(f"VRAM         : {VRAM_GB:.2f} GB")
print("Torch        :", torch.__version__)
print("CUDA runtime :", torch.version.cuda)
print("Ultralytics  :", ultralytics.__version__)
print("timm         :", timm.__version__)
print("SEG batch    :", SEG_BATCH)
print("CLS batch    :", CLS_BATCH)
print("STATUS: ENVIRONMENT_READY")


In [ ]:
# CELL 2 — PATHS + BEST EXISTING TEACHER CHECKPOINTS

PROJECT_ROOT = WORKSPACE / "data" / "Coral_Reef_AI_Project"

DET_ROOT = PROJECT_ROOT / "01_Data" / "Final" / "Detection_v1"

TRAIN_IMG = DET_ROOT / "images" / "train"
VAL_IMG = DET_ROOT / "images" / "validation"
TEST_IMG = DET_ROOT / "images" / "test"

TRAIN_COCO = DET_ROOT / "annotations" / "instances_train.json"
VAL_COCO = DET_ROOT / "annotations" / "instances_validation.json"

MANIFEST = (
    PROJECT_ROOT
    / "04_Dataset_Splits"
    / "classification_roi_manifest_split.csv"
)

MODELS_ROOT = WORKSPACE / "models"
MODELS_ROOT.mkdir(parents=True, exist_ok=True)

CACHE_ROOT = WORKSPACE / "cache"
CACHE_ROOT.mkdir(parents=True, exist_ok=True)

FINAL_ROOT = WORKSPACE / "outputs" / "CoralGuard_X_V2_FINAL"

LEGACY_MASK_ROOT = (
    WORKSPACE
    / "outputs"
    / "CoralGuard_X_FINAL"
    / "01_CoralSCOP_Masks"
)

NEW_MASK_ROOT = FINAL_ROOT / "01_Teacher_Masks"

# Reuse masks already generated by the previous notebook if present.
if (LEGACY_MASK_ROOT / "coralscop_mask_manifest.csv").exists():
    MASK_ROOT = LEGACY_MASK_ROOT
    print("Reusing existing CoralSCOP masks:", MASK_ROOT)
else:
    MASK_ROOT = NEW_MASK_ROOT

SEG_DATA_ROOT = CACHE_ROOT / "Coral_YOLO26m_SEG_Student_Dataset"
SEG_ROOT = FINAL_ROOT / "02_YOLO26m_SEG_Student"
DUAL_ROOT = FINAL_ROOT / "03_DualView_Data"
CLS_ROOT = FINAL_ROOT / "04_DualView_Classifier"
CAL_ROOT = FINAL_ROOT / "05_OOD_Calibration"
VIDEO_ROOT = FINAL_ROOT / "06_Final_Videos"

for d in [
    FINAL_ROOT,
    MASK_ROOT,
    SEG_ROOT,
    DUAL_ROOT,
    CLS_ROOT,
    CAL_ROOT,
    VIDEO_ROOT,
]:
    d.mkdir(parents=True, exist_ok=True)


def first_existing(paths):
    for p in paths:
        if p.exists():
            return p
    return None


# Optional manual override.
MANUAL_DETECTOR_TEACHER = None
MANUAL_CLASSIFIER_TEACHER = None


DETECTOR_TEACHER_CANDIDATES = [
    MODELS_ROOT / "best_yolo26m_plus150.pt",
    MODELS_ROOT / "best_yolo26m_extended.pt",
    MODELS_ROOT / "best.pt",
    WORKSPACE
    / "outputs"
    / "YOLO26m_EfficientNetV2S_FullPipeline"
    / "01_YOLO26m"
    / "Detection_YOLO26m"
    / "weights"
    / "best.pt",
]

CLASSIFIER_TEACHER_CANDIDATES = [
    MODELS_ROOT / "best_efficientnetv2s_plus150.pth",
    MODELS_ROOT / "best_efficientnetv2s_macro_f1.pth",
    WORKSPACE
    / "outputs"
    / "YOLO26m_EfficientNetV2S_FullPipeline"
    / "02_EfficientNetV2S"
    / "best_efficientnetv2s_macro_f1.pth",
]


DETECTOR_TEACHER = (
    Path(MANUAL_DETECTOR_TEACHER)
    if MANUAL_DETECTOR_TEACHER
    else first_existing(DETECTOR_TEACHER_CANDIDATES)
)

CLASSIFIER_TEACHER = (
    Path(MANUAL_CLASSIFIER_TEACHER)
    if MANUAL_CLASSIFIER_TEACHER
    else first_existing(CLASSIFIER_TEACHER_CANDIDATES)
)


if DETECTOR_TEACHER is None:
    raise FileNotFoundError(
        "No YOLO26m detector checkpoint found. "
        "Put the strongest best.pt in D:\\Coral_Local_Workspace\\models"
    )

if CLASSIFIER_TEACHER is None:
    raise FileNotFoundError(
        "No EfficientNetV2-S checkpoint found. "
        "Put the strongest .pth in D:\\Coral_Local_Workspace\\models"
    )


required = [
    TRAIN_IMG,
    VAL_IMG,
    TRAIN_COCO,
    VAL_COCO,
    MANIFEST,
    DETECTOR_TEACHER,
    CLASSIFIER_TEACHER,
]

for p in required:
    if not p.exists():
        raise FileNotFoundError(p)


print("=" * 110)
print("INPUTS")
print("=" * 110)
print("Detection teacher :", DETECTOR_TEACHER)
print("Classifier teacher:", CLASSIFIER_TEACHER)
print("Mask root         :", MASK_ROOT)
print("Final root        :", FINAL_ROOT)
print("Train             :", TRAIN_IMG)
print("Validation        :", VAL_IMG)
print("Test              : untouched")
print("STATUS: PATHS_READY")


In [ ]:
# CELL 3 — THIRD-PARTY SETUP: CORALSCOP + SAM2

THIRD_PARTY = WORKSPACE / "third_party"
THIRD_PARTY.mkdir(parents=True, exist_ok=True)

CORALSCOP_REPO = THIRD_PARTY / "CoralSCOP"
SAM2_REPO = THIRD_PARTY / "sam2"

GIT = shutil.which("git")

if GIT is None:
    raise RuntimeError(
        "Git for Windows is required and must be on PATH."
    )


if not CORALSCOP_REPO.exists():
    subprocess.check_call(
        [
            GIT,
            "clone",
            "https://github.com/zhengziqiang/CoralSCOP",
            str(CORALSCOP_REPO),
        ]
    )


if not SAM2_REPO.exists():
    subprocess.check_call(
        [
            GIT,
            "clone",
            "https://github.com/facebookresearch/sam2.git",
            str(SAM2_REPO),
        ]
    )


# Install SAM2 into the project venv only if import is unavailable.
try:
    import sam2
except Exception:
    print("Installing SAM2 into local .venv...")
    subprocess.check_call(
        [
            str(VENV_PYTHON),
            "-m",
            "pip",
            "install",
            "-e",
            str(SAM2_REPO),
        ]
    )


CORALSCOP_CKPT = MODELS_ROOT / "vit_b_coralscop.pth"

# Official checkpoint link used by the CoralSCOP project.
CORALSCOP_OFFICIAL_URL = (
    "https://www.dropbox.com/scl/fi/"
    "pw5jiq9oc8e8kvkx1fdk0/vit_b_coralscop.pth"
    "?rlkey=qczdohnzxwgwoadpzeht0lim2&dl=1"
)


if not CORALSCOP_CKPT.exists():
    import requests

    print("Downloading CoralSCOP ViT-B checkpoint...")

    with requests.get(
        CORALSCOP_OFFICIAL_URL,
        stream=True,
        timeout=120,
    ) as r:
        r.raise_for_status()

        with open(
            CORALSCOP_CKPT,
            "wb",
        ) as f:
            for chunk in r.iter_content(
                chunk_size=1024 * 1024
            ):
                if chunk:
                    f.write(chunk)


if (
    not CORALSCOP_CKPT.exists()
    or CORALSCOP_CKPT.stat().st_size
    < 50 * 1024 * 1024
):
    raise RuntimeError(
        "CoralSCOP checkpoint is missing/incomplete:\n"
        f"{CORALSCOP_CKPT}"
    )


print("CoralSCOP repo:", CORALSCOP_REPO)
print("SAM2 repo     :", SAM2_REPO)
print("CoralSCOP ckpt:", CORALSCOP_CKPT)
print("STATUS: THIRD_PARTY_READY")


In [ ]:
# CELL 4 — LOAD CORALSCOP MASK TEACHER

from pathlib import Path
import sys
import torch

repo_str = str(CORALSCOP_REPO)

if repo_str not in sys.path:
    sys.path.insert(
        0,
        repo_str,
    )

from segment_anything import (
    sam_model_registry,
    SamPredictor,
)


coralscop_model = sam_model_registry[
    "vit_b"
](
    checkpoint=str(
        CORALSCOP_CKPT
    )
)

coralscop_model.to(
    DEVICE
)

coralscop_model.eval()

coralscop_predictor = SamPredictor(
    coralscop_model
)


MASK_MIN_PIXELS = 40
MASK_MIN_SCORE = 0.35


print("CoralSCOP device:", next(coralscop_model.parameters()).device)
print("Predictor       :", type(coralscop_predictor).__name__)
print("MASK_MIN_SCORE  :", MASK_MIN_SCORE)
print("STATUS: CORALSCOP_TEACHER_READY")


In [ ]:
# CELL 5 — GENERATE / RESUME CORALSCOP PSEUDO MASKS FROM GT BOXES

from collections import defaultdict
from pathlib import Path
import json

import numpy as np
import pandas as pd
import cv2
import torch
from PIL import Image


if "coralscop_predictor" not in globals():
    raise RuntimeError(
        "Run CELL 4 first: CoralSCOP teacher is not loaded."
    )


MASK_MANIFEST = MASK_ROOT / "coralscop_mask_manifest.csv"


def resolve_image(
    folder,
    file_name,
):
    p = folder / Path(
        file_name
    ).name

    if p.exists():
        return p

    matches = list(
        folder.glob(
            Path(
                file_name
            ).stem
            + ".*"
        )
    )

    if not matches:
        raise FileNotFoundError(
            file_name
        )

    return matches[
        0
    ]


def box_area(
    box,
):
    x1, y1, x2, y2 = map(
        float,
        box,
    )

    return max(
        0.0,
        x2 - x1,
    ) * max(
        0.0,
        y2 - y1,
    )


def box_iou(
    a,
    b,
):
    ax1, ay1, ax2, ay2 = map(
        float,
        a,
    )

    bx1, by1, bx2, by2 = map(
        float,
        b,
    )

    ix1 = max(
        ax1,
        bx1,
    )

    iy1 = max(
        ay1,
        by1,
    )

    ix2 = min(
        ax2,
        bx2,
    )

    iy2 = min(
        ay2,
        by2,
    )

    inter = max(
        0.0,
        ix2 - ix1,
    ) * max(
        0.0,
        iy2 - iy1,
    )

    return inter / max(
        box_area(
            a
        )
        + box_area(
            b
        )
        - inter,
        1e-9,
    )


def mask_bbox(
    mask,
):
    ys, xs = np.where(
        mask
    )

    if len(
        xs
    ) == 0:
        return None

    return np.array(
        [
            xs.min(),
            ys.min(),
            xs.max() + 1,
            ys.max() + 1,
        ],
        dtype=np.float32,
    )


existing = {}

if MASK_MANIFEST.exists():
    old = pd.read_csv(
        MASK_MANIFEST
    )

    for _, r in old.iterrows():
        existing[
            (
                str(
                    r[
                        "split"
                    ]
                ),
                int(
                    r[
                        "ann_id"
                    ]
                ),
            )
        ] = dict(
            r
        )


rows = []


for (
    split,
    coco_path,
    img_folder,
) in [
    (
        "train",
        TRAIN_COCO,
        TRAIN_IMG,
    ),
    (
        "validation",
        VAL_COCO,
        VAL_IMG,
    ),
]:

    data = json.loads(
        coco_path.read_text(
            encoding="utf-8"
        )
    )

    by_img = defaultdict(
        list
    )

    for a in data[
        "annotations"
    ]:
        by_img[
            int(
                a[
                    "image_id"
                ]
            )
        ].append(
            a
        )

    info = {
        int(
            im[
                "id"
            ]
        ): im
        for im
        in data[
            "images"
        ]
    }

    out_dir = MASK_ROOT / split

    out_dir.mkdir(
        parents=True,
        exist_ok=True,
    )


    for n, (
        image_id,
        im,
    ) in enumerate(
        info.items(),
        start=1,
    ):

        anns = by_img[
            image_id
        ]


        if all(
            (
                split,
                int(
                    a[
                        "id"
                    ]
                ),
            )
            in existing
            for a
            in anns
        ):

            rows.extend(
                existing[
                    (
                        split,
                        int(
                            a[
                                "id"
                            ]
                        ),
                    )
                ]
                for a
                in anns
            )

            continue


        path = resolve_image(
            img_folder,
            im[
                "file_name"
            ],
        )

        rgb = np.asarray(
            Image.open(
                path
            ).convert(
                "RGB"
            )
        )

        H, W = rgb.shape[
            :2
        ]

        # One image embedding, many box prompts.
        coralscop_predictor.set_image(
            rgb
        )


        for a in anns:

            key = (
                split,
                int(
                    a[
                        "id"
                    ]
                ),
            )


            if key in existing:
                rows.append(
                    existing[
                        key
                    ]
                )
                continue


            x, y, w, h = map(
                float,
                a[
                    "bbox"
                ],
            )

            gt_box = np.array(
                [
                    x,
                    y,
                    x + w,
                    y + h,
                ],
                dtype=np.float32,
            )

            gt_box[
                0::2
            ] = np.clip(
                gt_box[
                    0::2
                ],
                0,
                W - 1,
            )

            gt_box[
                1::2
            ] = np.clip(
                gt_box[
                    1::2
                ],
                0,
                H - 1,
            )


            with torch.inference_mode():

                masks, scores, _ = coralscop_predictor.predict(
                    box=gt_box,
                    multimask_output=True,
                )


            order = np.argsort(
                scores
            )[
                ::-1
            ]


            chosen = None
            chosen_score = -1.0


            for idx in order:

                m = masks[
                    idx
                ].astype(
                    bool
                )

                if (
                    m.sum()
                    >= MASK_MIN_PIXELS
                ):
                    chosen = m
                    chosen_score = float(
                        scores[
                            idx
                        ]
                    )

                    if (
                        chosen_score
                        >= MASK_MIN_SCORE
                    ):
                        break


            if chosen is None:
                raise RuntimeError(
                    f"CoralSCOP returned no usable mask: "
                    f"{split} ann={a['id']}"
                )


            mb = mask_bbox(
                chosen
            )

            if mb is None:
                raise RuntimeError(
                    f"Empty mask: {split} ann={a['id']}"
                )


            mask_area = float(
                chosen.sum()
            )

            gt_area = max(
                box_area(
                    gt_box
                ),
                1.0,
            )

            area_ratio = (
                mask_area
                / gt_area
            )

            bbox_iou = box_iou(
                mb,
                gt_box,
            )


            mask_path = (
                out_dir
                / f"ann_{int(a['id']):06d}.png"
            )

            cv2.imwrite(
                str(
                    mask_path
                ),
                chosen.astype(
                    np.uint8
                )
                * 255,
            )


            row = {
                "split": split,
                "ann_id": int(
                    a[
                        "id"
                    ]
                ),
                "image_id": image_id,
                "image_path": str(
                    path
                ),
                "image_stem": path.stem,
                "mask_path": str(
                    mask_path
                ),
                "x1": float(
                    gt_box[
                        0
                    ]
                ),
                "y1": float(
                    gt_box[
                        1
                    ]
                ),
                "x2": float(
                    gt_box[
                        2
                    ]
                ),
                "y2": float(
                    gt_box[
                        3
                    ]
                ),
                "mask_score": chosen_score,
                "mask_area_ratio_to_gt_box": area_ratio,
                "mask_bbox_iou_gt_box": bbox_iou,
            }


            rows.append(
                row
            )

            existing[
                key
            ] = row


        pd.DataFrame(
            rows
        ).drop_duplicates(
            [
                "split",
                "ann_id",
            ]
        ).to_csv(
            MASK_MANIFEST,
            index=False,
        )


        if (
            n
            % 25
            == 0
        ):
            print(
                f"{split}: {n}/{len(info)}"
            )


mask_df = pd.read_csv(
    MASK_MANIFEST
)


print()
print(
    mask_df.groupby(
        "split"
    ).size()
)

print(
    "Mean CoralSCOP score:",
    float(
        mask_df[
            "mask_score"
        ].mean()
    ),
)

print(
    "Manifest:",
    MASK_MANIFEST,
)

print("STATUS: TEACHER_MASKS_READY")


In [ ]:
# CELL 6 — BUILD YOLO26m-SEG STUDENT DATASET FROM TEACHER MASKS

if SEG_DATA_ROOT.exists():
    shutil.rmtree(
        SEG_DATA_ROOT
    )

for split in [
    "train",
    "validation",
]:
    (
        SEG_DATA_ROOT
        / "images"
        / split
    ).mkdir(
        parents=True,
        exist_ok=True,
    )

    (
        SEG_DATA_ROOT
        / "labels"
        / split
    ).mkdir(
        parents=True,
        exist_ok=True,
    )


# Quality filter for pseudo masks.
PSEUDO_MIN_SCORE = 0.35
PSEUDO_MIN_AREA_RATIO = 0.05
PSEUDO_MAX_AREA_RATIO = 2.50
PSEUDO_MIN_BBOX_IOU = 0.10

MIN_CONTOUR_PIXELS = 40


def mask_quality(
    row,
    mask,
):
    gt_box = [
        float(
            row[
                "x1"
            ]
        ),
        float(
            row[
                "y1"
            ]
        ),
        float(
            row[
                "x2"
            ]
        ),
        float(
            row[
                "y2"
            ]
        ),
    ]

    mb = mask_bbox(
        mask
    )

    if mb is None:
        return (
            False,
            {
                "area_ratio": 0.0,
                "bbox_iou": 0.0,
            },
        )

    area_ratio = float(
        mask.sum()
    ) / max(
        box_area(
            gt_box
        ),
        1.0,
    )

    bbox_iou = box_iou(
        mb,
        gt_box,
    )

    score = float(
        row.get(
            "mask_score",
            0.0,
        )
    )

    ok = (
        score
        >= PSEUDO_MIN_SCORE
        and area_ratio
        >= PSEUDO_MIN_AREA_RATIO
        and area_ratio
        <= PSEUDO_MAX_AREA_RATIO
        and bbox_iou
        >= PSEUDO_MIN_BBOX_IOU
        and int(
            mask.sum()
        )
        >= MIN_CONTOUR_PIXELS
    )

    return (
        bool(
            ok
        ),
        {
            "area_ratio": area_ratio,
            "bbox_iou": bbox_iou,
            "score": score,
        },
    )


def mask_to_yolo_polygon(
    mask,
):
    u8 = (
        mask.astype(
            np.uint8
        )
        * 255
    )

    contours, _ = cv2.findContours(
        u8,
        cv2.RETR_EXTERNAL,
        cv2.CHAIN_APPROX_NONE,
    )

    if not contours:
        return None

    # One YOLO row = one instance polygon.
    # Use the largest connected external contour for this coral instance.
    contour = max(
        contours,
        key=cv2.contourArea,
    )

    if cv2.contourArea(
        contour
    ) < MIN_CONTOUR_PIXELS:
        return None

    perimeter = cv2.arcLength(
        contour,
        True,
    )

    epsilon = max(
        1.0,
        0.0025
        * perimeter,
    )

    poly = cv2.approxPolyDP(
        contour,
        epsilon,
        True,
    ).reshape(
        -1,
        2,
    )

    if len(
        poly
    ) < 3:
        return None

    return poly.astype(
        np.float32
    )


def coco_image_info(
    coco_path,
):
    data = json.loads(
        coco_path.read_text(
            encoding="utf-8"
        )
    )

    return {
        int(
            im[
                "id"
            ]
        ): im
        for im
        in data[
            "images"
        ]
    }


image_info_by_split = {
    "train": coco_image_info(
        TRAIN_COCO
    ),
    "validation": coco_image_info(
        VAL_COCO
    ),
}

folder_by_split = {
    "train": TRAIN_IMG,
    "validation": VAL_IMG,
}


audit_rows = []


for split in [
    "train",
    "validation",
]:

    sdf = mask_df[
        mask_df[
            "split"
        ].astype(
            str
        )
        == split
    ].copy()

    grouped = sdf.groupby(
        "image_id"
    )


    for image_id, group in grouped:

        image_id = int(
            image_id
        )

        im_info = image_info_by_split[
            split
        ][
            image_id
        ]

        src = resolve_image(
            folder_by_split[
                split
            ],
            im_info[
                "file_name"
            ],
        )

        with Image.open(
            src
        ) as im:
            W, H = im.size


        dst_img = (
            SEG_DATA_ROOT
            / "images"
            / split
            / src.name
        )

        shutil.copy2(
            src,
            dst_img,
        )


        label_lines = []


        for _, row in group.iterrows():

            mask = (
                cv2.imread(
                    str(
                        row[
                            "mask_path"
                        ]
                    ),
                    cv2.IMREAD_GRAYSCALE,
                )
                > 127
            )

            ok, q = mask_quality(
                row,
                mask,
            )

            polygon = (
                mask_to_yolo_polygon(
                    mask
                )
                if ok
                else None
            )


            keep = (
                ok
                and polygon
                is not None
            )


            audit_rows.append(
                {
                    "split": split,
                    "ann_id": int(
                        row[
                            "ann_id"
                        ]
                    ),
                    "image_id": image_id,
                    "mask_score": q[
                        "score"
                    ],
                    "area_ratio": q[
                        "area_ratio"
                    ],
                    "bbox_iou": q[
                        "bbox_iou"
                    ],
                    "kept_for_student": bool(
                        keep
                    ),
                }
            )


            if not keep:
                continue


            polygon[
                :,
                0
            ] = np.clip(
                polygon[
                    :,
                    0
                ]
                / float(
                    W
                ),
                0.0,
                1.0,
            )

            polygon[
                :,
                1
            ] = np.clip(
                polygon[
                    :,
                    1
                ]
                / float(
                    H
                ),
                0.0,
                1.0,
            )


            coords = " ".join(
                f"{float(v):.7f}"
                for v
                in polygon.reshape(
                    -1
                )
            )

            label_lines.append(
                "0 "
                + coords
            )


        label_path = (
            SEG_DATA_ROOT
            / "labels"
            / split
            / f"{src.stem}.txt"
        )

        label_path.write_text(
            "\n".join(
                label_lines
            ),
            encoding="utf-8",
        )


audit_df = pd.DataFrame(
    audit_rows
)

AUDIT_CSV = (
    SEG_ROOT
    / "teacher_mask_quality_audit.csv"
)

audit_df.to_csv(
    AUDIT_CSV,
    index=False,
)


kept_counts = audit_df.groupby(
    "split"
)[
    "kept_for_student"
].agg(
    [
        "sum",
        "count",
        "mean",
    ]
)


print(
    kept_counts
)


# Hard safety: if the teacher-mask filter removes too much data,
# stop instead of silently training on a broken pseudo-label set.
for split in [
    "train",
    "validation",
]:

    retain = float(
        kept_counts.loc[
            split,
            "mean"
        ]
    )

    if retain < 0.75:
        raise RuntimeError(
            f"Only {retain:.1%} of {split} masks survived the quality filter. "
            "Inspect CoralSCOP masks before training the student."
        )


SEG_YAML = (
    SEG_DATA_ROOT
    / "data.yaml"
)

SEG_YAML.write_text(
    "path: "
    + str(
        SEG_DATA_ROOT
    ).replace(
        "\\",
        "/",
    )
    + "\n"
    + "train: images/train\n"
    + "val: images/validation\n"
    + "names:\n"
    + "  0: target_coral\n",
    encoding="utf-8",
)


print("SEG dataset:", SEG_DATA_ROOT)
print("SEG yaml   :", SEG_YAML)
print("Audit      :", AUDIT_CSV)
print("STATUS: STUDENT_SEG_DATA_READY")


In [ ]:
# CELL 7 — INITIALIZE YOLO26m-SEG STUDENT FROM TWO SOURCES

# Source A:
# official segmentation architecture/segmentation-specific weights.
#
# Source B:
# your trained coral YOLO26m detector. Ultralytics load() transfers
# matching tensors by name+shape, so compatible coral backbone/neck/
# box-regression knowledge is reused while segmentation-only layers remain.

SEG_OFFICIAL_LOCAL = MODELS_ROOT / "yolo26m-seg.pt"

SEG_BASE = (
    str(
        SEG_OFFICIAL_LOCAL
    )
    if SEG_OFFICIAL_LOCAL.exists()
    else "yolo26m-seg.pt"
)


print("SEG base           :", SEG_BASE)
print("Detection teacher  :", DETECTOR_TEACHER)


student = YOLO(
    SEG_BASE
)


# Preserve the segmentation architecture, then inject every compatible
# weight tensor from the coral detection teacher.
student.load(
    str(
        DETECTOR_TEACHER
    )
)


print()
print("Student task:", student.task)
print("Student info:")
student.info()

print("STATUS: SEG_STUDENT_INITIALIZED")


In [ ]:
# CELL 8 — TRAIN YOLO26m-SEG STUDENT FOR 150 EPOCHS

SEG_EPOCHS = 150
SEG_IMGSZ = 1024

SEG_RUN_NAME = "YOLO26m_SEG_TeacherStudent_150ep"


print("=" * 115)
print("YOLO26m-SEG STUDENT TRAINING")
print("=" * 115)
print("Teacher masks       : CoralSCOP")
print("Weight teacher      :", DETECTOR_TEACHER)
print("Student             : YOLO26m-seg")
print("Epochs              :", SEG_EPOCHS)
print("Image size          :", SEG_IMGSZ)
print("Batch               :", SEG_BATCH)
print("Train test split    : NO")
print("=" * 115)


t0 = time.time()


student.train(
    data=str(
        SEG_YAML
    ),

    epochs=SEG_EPOCHS,

    imgsz=SEG_IMGSZ,

    batch=SEG_BATCH,

    device=0,

    # Safe for Windows notebooks.
    workers=0,

    optimizer="AdamW",

    lr0=1e-4,

    lrf=0.02,

    weight_decay=5e-4,

    cos_lr=True,

    warmup_epochs=1.0,

    patience=40,

    save=True,

    save_period=10,

    seed=SEED,

    deterministic=True,

    # Underwater-friendly but morphology-preserving.
    hsv_h=0.010,

    hsv_s=0.18,

    hsv_v=0.16,

    degrees=5.0,

    translate=0.05,

    scale=0.12,

    shear=0.0,

    perspective=0.0,

    fliplr=0.50,

    flipud=0.20,

    # Moderate small/crowded support.
    mosaic=0.15,

    close_mosaic=15,

    mixup=0.0,

    # Segmentation-specific augmentation.
    copy_paste=0.10,

    copy_paste_mode="flip",

    multi_scale=0.10,

    amp=True,

    max_det=500,

    project=str(
        SEG_ROOT
    ),

    name=SEG_RUN_NAME,

    exist_ok=True,

    plots=True,

    verbose=True,
)


SEG_RUN_DIR = (
    SEG_ROOT
    / SEG_RUN_NAME
)

SEG_BEST = (
    SEG_RUN_DIR
    / "weights"
    / "best.pt"
)

SEG_LAST = (
    SEG_RUN_DIR
    / "weights"
    / "last.pt"
)


if not SEG_BEST.exists():
    raise FileNotFoundError(
        SEG_BEST
    )


print()
print(
    f"Training minutes: {(time.time()-t0)/60:.2f}"
)

print("Best:", SEG_BEST)
print("Last:", SEG_LAST)
print("STATUS: SEG_STUDENT_TRAINED")


In [ ]:
# CELL 9 — VALIDATE SEGMENTATION STUDENT + ORIGINAL-GT DETECTION CONFIDENCE SWEEP

seg_model = YOLO(
    str(
        SEG_BEST
    )
)


seg_metrics = seg_model.val(
    data=str(
        SEG_YAML
    ),
    split="val",
    imgsz=SEG_IMGSZ,
    batch=SEG_BATCH,
    conf=0.001,
    iou=0.70,
    max_det=500,
    device=0,
    workers=0,
    plots=True,
    verbose=True,
)


SEG_METRICS = {
    "box_mAP50_95_pseudo": float(
        seg_metrics.box.map
    ),
    "box_mAP50_pseudo": float(
        seg_metrics.box.map50
    ),
    "box_mAP75_pseudo": float(
        seg_metrics.box.map75
    ),
    "mask_mAP50_95_pseudo": float(
        seg_metrics.seg.map
    ),
    "mask_mAP50_pseudo": float(
        seg_metrics.seg.map50
    ),
    "mask_mAP75_pseudo": float(
        seg_metrics.seg.map75
    ),
}


print("=" * 110)
print("PSEUDO-MASK VALIDATION")
print("=" * 110)

for k, v in SEG_METRICS.items():
    print(
        f"{k:28s}: {v:.6f}"
    )


# -----------------------------------------------------------------
# Operational detector threshold is tuned against the ORIGINAL
# validation GT boxes, not against pseudo-mask-derived boxes.
# -----------------------------------------------------------------

val_coco = json.loads(
    VAL_COCO.read_text(
        encoding="utf-8"
    )
)

val_info = {
    int(
        im[
            "id"
        ]
    ): im
    for im
    in val_coco[
        "images"
    ]
}

val_gt_by_stem = defaultdict(
    list
)

for a in val_coco[
    "annotations"
]:
    im = val_info[
        int(
            a[
                "image_id"
            ]
        )
    ]

    stem = Path(
        im[
            "file_name"
        ]
    ).stem

    x, y, w, h = map(
        float,
        a[
            "bbox"
        ],
    )

    val_gt_by_stem[
        stem
    ].append(
        [
            x,
            y,
            x + w,
            y + h,
        ]
    )


def iou_matrix(
    a,
    b,
):
    a = np.asarray(
        a,
        dtype=np.float32,
    ).reshape(
        -1,
        4,
    )

    b = np.asarray(
        b,
        dtype=np.float32,
    ).reshape(
        -1,
        4,
    )

    if (
        len(
            a
        ) == 0
        or len(
            b
        ) == 0
    ):
        return np.zeros(
            (
                len(
                    a
                ),
                len(
                    b
                ),
            ),
            dtype=np.float32,
        )

    lt = np.maximum(
        a[
            :,
            None,
            :2,
        ],
        b[
            None,
            :,
            :2,
        ],
    )

    rb = np.minimum(
        a[
            :,
            None,
            2:,
        ],
        b[
            None,
            :,
            2:,
        ],
    )

    wh = np.clip(
        rb - lt,
        0,
        None,
    )

    inter = (
        wh[
            ...,
            0
        ]
        * wh[
            ...,
            1
        ]
    )

    aa = (
        np.clip(
            a[
                :,
                2
            ]
            - a[
                :,
                0
            ],
            0,
            None,
        )
        * np.clip(
            a[
                :,
                3
            ]
            - a[
                :,
                1
            ],
            0,
            None,
        )
    )

    bb = (
        np.clip(
            b[
                :,
                2
            ]
            - b[
                :,
                0
            ],
            0,
            None,
        )
        * np.clip(
            b[
                :,
                3
            ]
            - b[
                :,
                1
            ],
            0,
            None,
        )
    )

    return inter / np.clip(
        aa[
            :,
            None
        ]
        + bb[
            None,
            :
        ]
        - inter,
        1e-9,
        None,
    )


pred_cache = []


val_images = sorted(
    [
        p
        for p
        in VAL_IMG.iterdir()
        if p.is_file()
        and p.suffix.lower()
        in {
            ".jpg",
            ".jpeg",
            ".png",
            ".bmp",
            ".webp",
        }
    ]
)


for n, path in enumerate(
    val_images,
    start=1,
):

    r = seg_model.predict(
        source=str(
            path
        ),
        imgsz=SEG_IMGSZ,
        conf=0.01,
        iou=0.70,
        max_det=500,
        device=0,
        verbose=False,
    )[
        0
    ]

    boxes = (
        r.boxes.xyxy
        .detach()
        .cpu()
        .numpy()
        .astype(
            np.float32
        )
        if r.boxes
        is not None
        else np.empty(
            (
                0,
                4,
            ),
            dtype=np.float32,
        )
    )

    confs = (
        r.boxes.conf
        .detach()
        .cpu()
        .numpy()
        .astype(
            np.float32
        )
        if r.boxes
        is not None
        else np.empty(
            0,
            dtype=np.float32,
        )
    )

    gt = np.asarray(
        val_gt_by_stem[
            path.stem
        ],
        dtype=np.float32,
    ).reshape(
        -1,
        4,
    )

    pred_cache.append(
        (
            boxes,
            confs,
            gt,
        )
    )

    if (
        n
        % 25
        == 0
    ):
        print(
            f"Cached {n}/{len(val_images)} validation images"
        )


def score_confidence(
    threshold,
    match_iou=0.50,
):
    tp = 0
    fp = 0
    fn = 0

    for boxes, confs, gt in pred_cache:

        keep = confs >= threshold

        pb = boxes[
            keep
        ]

        if (
            len(
                pb
            ) == 0
        ):
            fn += len(
                gt
            )
            continue

        if (
            len(
                gt
            ) == 0
        ):
            fp += len(
                pb
            )
            continue

        mat = iou_matrix(
            pb,
            gt,
        )

        rr, cc = linear_sum_assignment(
            1.0
            - mat
        )

        matched = 0

        for r_idx, c_idx in zip(
            rr,
            cc,
        ):
            if (
                mat[
                    r_idx,
                    c_idx
                ]
                >= match_iou
            ):
                matched += 1

        tp += matched

        fp += len(
            pb
        ) - matched

        fn += len(
            gt
        ) - matched

    precision = tp / max(
        tp + fp,
        1,
    )

    recall = tp / max(
        tp + fn,
        1,
    )

    f1 = (
        2
        * precision
        * recall
        / max(
            precision
            + recall,
            1e-9,
        )
    )

    return {
        "confidence": float(
            threshold
        ),
        "tp": int(
            tp
        ),
        "fp": int(
            fp
        ),
        "fn": int(
            fn
        ),
        "precision": float(
            precision
        ),
        "recall": float(
            recall
        ),
        "f1": float(
            f1
        ),
    }


sweep = [
    score_confidence(
        c
    )
    for c
    in np.arange(
        0.05,
        0.751,
        0.025,
    )
]


sweep_df = pd.DataFrame(
    sweep
)

best_det = sweep_df.loc[
    sweep_df[
        "f1"
    ].idxmax()
]


BEST_SEG_CONF = float(
    best_det[
        "confidence"
    ]
)


SWEEP_CSV = (
    SEG_ROOT
    / "student_confidence_sweep_original_GT.csv"
)

sweep_df.to_csv(
    SWEEP_CSV,
    index=False,
)


PROMOTED_SEG = (
    MODELS_ROOT
    / "best_yolo26m_seg_student.pt"
)

shutil.copy2(
    SEG_BEST,
    PROMOTED_SEG,
)


OP_CONFIG = {
    "student_checkpoint": str(
        PROMOTED_SEG
    ),
    "best_seg_conf_original_gt": BEST_SEG_CONF,
    "original_gt_precision": float(
        best_det[
            "precision"
        ]
    ),
    "original_gt_recall": float(
        best_det[
            "recall"
        ]
    ),
    "original_gt_f1": float(
        best_det[
            "f1"
        ]
    ),
    "pseudo_mask_validation": SEG_METRICS,
    "held_out_test_used": False,
}


OP_CONFIG_JSON = (
    SEG_ROOT
    / "student_operating_config.json"
)

OP_CONFIG_JSON.write_text(
    json.dumps(
        OP_CONFIG,
        indent=2,
    ),
    encoding="utf-8",
)


print()
print("=" * 110)
print("ORIGINAL GT OPERATING POINT")
print("=" * 110)
print(best_det.to_dict())
print("Promoted student:", PROMOTED_SEG)
print("Config          :", OP_CONFIG_JSON)
print("STATUS: SEG_STUDENT_VALIDATED")


In [ ]:
# CELL 10 — BUILD DUAL-VIEW CLASSIFIER DATA FROM TEACHER MASKS

roi = pd.read_csv(
    MANIFEST
)

needed = {
    "roi_id",
    "image_stem",
    "raw_detection_label",
    "x1_px",
    "y1_px",
    "x2_px",
    "y2_px",
    "split",
}

missing = needed - set(
    roi.columns
)

if missing:
    raise RuntimeError(
        "Classification manifest schema mismatch: "
        + str(
            sorted(
                missing
            )
        )
    )


mask_index = defaultdict(
    list
)

for _, r in mask_df.iterrows():
    mask_index[
        (
            str(
                r[
                    "split"
                ]
            ),
            str(
                r[
                    "image_stem"
                ]
            ),
        )
    ].append(
        r
    )


positive_rows = []


for _, r in roi[
    roi[
        "split"
    ].astype(
        str
    ).isin(
        [
            "train",
            "validation",
        ]
    )
].iterrows():

    key = (
        str(
            r[
                "split"
            ]
        ),
        str(
            r[
                "image_stem"
            ]
        ),
    )

    candidates = mask_index[
        key
    ]

    q = [
        float(
            r[
                "x1_px"
            ]
        ),
        float(
            r[
                "y1_px"
            ]
        ),
        float(
            r[
                "x2_px"
            ]
        ),
        float(
            r[
                "y2_px"
            ]
        ),
    ]

    best = max(
        candidates,
        key=lambda z: box_iou(
            q,
            [
                z[
                    "x1"
                ],
                z[
                    "y1"
                ],
                z[
                    "x2"
                ],
                z[
                    "y2"
                ],
            ],
        ),
    )

    mapped_iou = box_iou(
        q,
        [
            best[
                "x1"
            ],
            best[
                "y1"
            ],
            best[
                "x2"
            ],
            best[
                "y2"
            ],
        ],
    )

    if mapped_iou < 0.90:
        raise RuntimeError(
            f"Mask mapping failed for {r['roi_id']}: {mapped_iou:.3f}"
        )

    positive_rows.append(
        {
            "roi_id": str(
                r[
                    "roi_id"
                ]
            ),
            "split": str(
                r[
                    "split"
                ]
            ),
            "label": str(
                r[
                    "raw_detection_label"
                ]
            ),
            "image_path": str(
                best[
                    "image_path"
                ]
            ),
            "mask_path": str(
                best[
                    "mask_path"
                ]
            ),
            "x1": q[
                0
            ],
            "y1": q[
                1
            ],
            "x2": q[
                2
            ],
            "y2": q[
                3
            ],
        }
    )


pos = pd.DataFrame(
    positive_rows
)


assert (
    pos[
        "split"
    ]
    == "train"
).sum() == 1540

assert (
    pos[
        "split"
    ]
    == "validation"
).sum() == 354


CONTEXT_PAD = 0.10

PAIR_MANIFEST = (
    DUAL_ROOT
    / "dual_view_manifest.csv"
)


def padded_box(
    box,
    W,
    H,
    pad=0.10,
):
    x1, y1, x2, y2 = map(
        float,
        box,
    )

    bw = max(
        1.0,
        x2 - x1,
    )

    bh = max(
        1.0,
        y2 - y1,
    )

    px = bw * pad
    py = bh * pad

    return (
        max(
            0,
            int(
                math.floor(
                    x1 - px
                )
            ),
        ),
        max(
            0,
            int(
                math.floor(
                    y1 - py
                )
            ),
        ),
        min(
            W,
            int(
                math.ceil(
                    x2 + px
                )
            ),
        ),
        min(
            H,
            int(
                math.ceil(
                    y2 + py
                )
            ),
        ),
    )


pair_rows = []


for _, r in pos.iterrows():

    out_dir = (
        DUAL_ROOT
        / r[
            "split"
        ]
        / r[
            "label"
        ]
    )

    out_dir.mkdir(
        parents=True,
        exist_ok=True,
    )

    cpath = (
        out_dir
        / f"{r['roi_id']}__context.jpg"
    )

    mpath = (
        out_dir
        / f"{r['roi_id']}__masked.jpg"
    )


    if not (
        cpath.exists()
        and mpath.exists()
    ):

        rgb = np.asarray(
            Image.open(
                r[
                    "image_path"
                ]
            ).convert(
                "RGB"
            )
        )

        mask = (
            cv2.imread(
                r[
                    "mask_path"
                ],
                cv2.IMREAD_GRAYSCALE,
            )
            > 127
        )

        H, W = rgb.shape[
            :2
        ]

        x1, y1, x2, y2 = padded_box(
            [
                r[
                    "x1"
                ],
                r[
                    "y1"
                ],
                r[
                    "x2"
                ],
                r[
                    "y2"
                ],
            ],
            W,
            H,
            CONTEXT_PAD,
        )

        context = rgb[
            y1:y2,
            x1:x2,
        ].copy()

        local_mask = mask[
            y1:y2,
            x1:x2,
        ]

        masked = context.copy()

        masked[
            ~local_mask
        ] = 0


        Image.fromarray(
            context
        ).save(
            cpath,
            "JPEG",
            quality=95,
            subsampling=0,
        )

        Image.fromarray(
            masked
        ).save(
            mpath,
            "JPEG",
            quality=95,
            subsampling=0,
        )


    pair_rows.append(
        {
            "split": r[
                "split"
            ],
            "label": r[
                "label"
            ],
            "context_path": str(
                cpath
            ),
            "masked_path": str(
                mpath
            ),
            "source": "CORAL",
        }
    )


# ------------------------------------------------------------
# Real NON_CORAL background negatives outside GT boxes.
# ------------------------------------------------------------

def coco_boxes(
    coco_path,
):
    data = json.loads(
        coco_path.read_text(
            encoding="utf-8"
        )
    )

    info = {
        int(
            im[
                "id"
            ]
        ): im
        for im
        in data[
            "images"
        ]
    }

    boxes = defaultdict(
        list
    )

    for a in data[
        "annotations"
    ]:
        x, y, w, h = map(
            float,
            a[
                "bbox"
            ],
        )

        boxes[
            int(
                a[
                    "image_id"
                ]
            )
        ].append(
            [
                x,
                y,
                x + w,
                y + h,
            ]
        )

    return (
        info,
        boxes,
    )


rng = np.random.default_rng(
    SEED
)


for (
    split,
    coco_path,
    img_folder,
) in [
    (
        "train",
        TRAIN_COCO,
        TRAIN_IMG,
    ),
    (
        "validation",
        VAL_COCO,
        VAL_IMG,
    ),
]:

    info, boxes_by_img = coco_boxes(
        coco_path
    )

    out_dir = (
        DUAL_ROOT
        / split
        / "NON_CORAL"
    )

    out_dir.mkdir(
        parents=True,
        exist_ok=True,
    )


    for image_id, im in info.items():

        path = resolve_image(
            img_folder,
            im[
                "file_name"
            ],
        )

        rgb = np.asarray(
            Image.open(
                path
            ).convert(
                "RGB"
            )
        )

        H, W = rgb.shape[
            :2
        ]

        q = None


        for _ in range(
            150
        ):

            side = int(
                max(
                    48,
                    min(
                        W,
                        H,
                    )
                    * float(
                        rng.uniform(
                            0.10,
                            0.32,
                        )
                    ),
                )
            )

            if (
                W <= side
                or H <= side
            ):
                continue

            x1 = int(
                rng.integers(
                    0,
                    W - side,
                )
            )

            y1 = int(
                rng.integers(
                    0,
                    H - side,
                )
            )

            cand = [
                x1,
                y1,
                x1 + side,
                y1 + side,
            ]

            mx = max(
                [
                    box_iou(
                        cand,
                        b,
                    )
                    for b
                    in boxes_by_img[
                        image_id
                    ]
                ]
                + [
                    0.0
                ]
            )

            if mx < 0.02:
                q = cand
                break


        if q is None:
            continue


        x1, y1, x2, y2 = q

        context = rgb[
            y1:y2,
            x1:x2,
        ].copy()

        masked = np.zeros_like(
            context
        )

        stem = Path(
            im[
                "file_name"
            ]
        ).stem

        cpath = (
            out_dir
            / f"{stem}__neg__context.jpg"
        )

        mpath = (
            out_dir
            / f"{stem}__neg__masked.jpg"
        )


        Image.fromarray(
            context
        ).save(
            cpath,
            "JPEG",
            quality=95,
            subsampling=0,
        )

        Image.fromarray(
            masked
        ).save(
            mpath,
            "JPEG",
            quality=95,
            subsampling=0,
        )


        pair_rows.append(
            {
                "split": split,
                "label": "NON_CORAL",
                "context_path": str(
                    cpath
                ),
                "masked_path": str(
                    mpath
                ),
                "source": "BACKGROUND_NEGATIVE",
            }
        )


pair_df = pd.DataFrame(
    pair_rows
)

pair_df.to_csv(
    PAIR_MANIFEST,
    index=False,
)


print(
    pair_df.groupby(
        [
            "split",
            "label",
        ]
    ).size()
)

print("Manifest:", PAIR_MANIFEST)
print("STATUS: DUAL_VIEW_DATA_READY")


In [ ]:
# CELL 11 — DUAL-VIEW EFFICIENTNETV2-S + TRANSFER LEARNING

old_ckpt = torch.load(
    CLASSIFIER_TEACHER,
    map_location=DEVICE,
    weights_only=False,
)


MODEL_NAME = old_ckpt.get(
    "model_name",
    "tf_efficientnetv2_s.in21k_ft_in1k",
)

OLD_CLASS_TO_IDX = old_ckpt[
    "class_to_idx"
]

KNOWN_CLASSES = [
    x
    for x, _
    in sorted(
        OLD_CLASS_TO_IDX.items(),
        key=lambda kv: kv[
            1
        ],
    )
]

ALL_CLASSES = (
    KNOWN_CLASSES
    + [
        "NON_CORAL"
    ]
)

CLASS_TO_IDX = {
    c: i
    for i, c
    in enumerate(
        ALL_CLASSES
    )
}

IDX_TO_CLASS = {
    i: c
    for c, i
    in CLASS_TO_IDX.items()
}

NCLS = len(
    ALL_CLASSES
)


CLS_SIZE = int(
    old_ckpt.get(
        "input_size",
        300,
    )
)

MEAN = tuple(
    old_ckpt.get(
        "mean",
        (
            0.5,
            0.5,
            0.5,
        ),
    )
)

STD = tuple(
    old_ckpt.get(
        "std",
        (
            0.5,
            0.5,
            0.5,
        ),
    )
)


train_tf = transforms.Compose(
    [
        transforms.Resize(
            (
                CLS_SIZE,
                CLS_SIZE,
            ),
            interpolation=InterpolationMode.BICUBIC,
        ),
        transforms.RandomHorizontalFlip(
            0.5
        ),
        transforms.RandomVerticalFlip(
            0.08
        ),
        transforms.RandomRotation(
            4,
            interpolation=InterpolationMode.BICUBIC,
        ),
        transforms.ColorJitter(
            brightness=0.10,
            contrast=0.10,
            saturation=0.06,
            hue=0.02,
        ),
        transforms.ToTensor(),
        transforms.Normalize(
            MEAN,
            STD,
        ),
    ]
)


val_tf = transforms.Compose(
    [
        transforms.Resize(
            (
                CLS_SIZE,
                CLS_SIZE,
            ),
            interpolation=InterpolationMode.BICUBIC,
        ),
        transforms.ToTensor(),
        transforms.Normalize(
            MEAN,
            STD,
        ),
    ]
)


class DualPairDataset(
    Dataset
):
    def __init__(
        self,
        df,
        transform,
    ):
        self.df = df.reset_index(
            drop=True
        )

        self.transform = transform

    def __len__(
        self
    ):
        return len(
            self.df
        )

    def __getitem__(
        self,
        idx,
    ):
        r = self.df.iloc[
            idx
        ]

        context = Image.open(
            r[
                "context_path"
            ]
        ).convert(
            "RGB"
        )

        masked = Image.open(
            r[
                "masked_path"
            ]
        ).convert(
            "RGB"
        )

        context = self.transform(
            context
        )

        masked = self.transform(
            masked
        )

        y = CLASS_TO_IDX[
            str(
                r[
                    "label"
                ]
            )
        ]

        return (
            context,
            masked,
            y,
        )


train_pairs = pair_df[
    pair_df[
        "split"
    ]
    == "train"
].copy()

val_pairs = pair_df[
    pair_df[
        "split"
    ]
    == "validation"
].copy()


train_ds = DualPairDataset(
    train_pairs,
    train_tf,
)

train_eval_ds = DualPairDataset(
    train_pairs,
    val_tf,
)

val_ds = DualPairDataset(
    val_pairs,
    val_tf,
)


labels_train = np.asarray(
    [
        CLASS_TO_IDX[
            str(
                x
            )
        ]
        for x
        in train_pairs[
            "label"
        ].tolist()
    ],
    dtype=np.int64,
)


counts = np.bincount(
    labels_train,
    minlength=NCLS,
).astype(
    np.float32
)

class_weights_np = np.sqrt(
    counts.max()
    / np.maximum(
        counts,
        1.0,
    )
)

class_weights_np /= class_weights_np.mean()


sample_weights = class_weights_np[
    labels_train
]


sampler = WeightedRandomSampler(
    weights=torch.tensor(
        sample_weights,
        dtype=torch.double,
    ),
    num_samples=len(
        sample_weights
    ),
    replacement=True,
)


train_loader = DataLoader(
    train_ds,
    batch_size=CLS_BATCH,
    sampler=sampler,
    num_workers=0,
    pin_memory=True,
)

train_eval_loader = DataLoader(
    train_eval_ds,
    batch_size=CLS_BATCH,
    shuffle=False,
    num_workers=0,
    pin_memory=True,
)

val_loader = DataLoader(
    val_ds,
    batch_size=CLS_BATCH,
    shuffle=False,
    num_workers=0,
    pin_memory=True,
)


class DualViewEfficientNet(
    nn.Module
):
    def __init__(
        self,
        model_name,
        num_classes,
    ):
        super().__init__()

        self.backbone = timm.create_model(
            model_name,
            pretrained=False,
            num_classes=0,
            global_pool="avg",
        )

        feat_dim = int(
            self.backbone.num_features
        )

        self.fusion = nn.Sequential(
            nn.Linear(
                feat_dim
                * 2,
                feat_dim,
            ),
            nn.GELU(),
            nn.LayerNorm(
                feat_dim
            ),
            nn.Dropout(
                0.20
            ),
        )

        self.head = nn.Linear(
            feat_dim,
            num_classes,
        )

        self.feat_dim = feat_dim

    def forward(
        self,
        context,
        masked,
        return_embedding=False,
    ):
        f_context = self.backbone(
            context
        )

        f_masked = self.backbone(
            masked
        )

        fused = torch.cat(
            [
                f_context,
                f_masked,
            ],
            dim=1,
        )

        emb = self.fusion(
            fused
        )

        logits = self.head(
            emb
        )

        if return_embedding:
            return (
                logits,
                emb,
            )

        return logits


dual_model = DualViewEfficientNet(
    MODEL_NAME,
    NCLS,
).to(
    DEVICE
)


# Transfer all matching backbone tensors from the existing classifier.
old_state = old_ckpt[
    "model_state_dict"
]

new_backbone_state = (
    dual_model.backbone.state_dict()
)

transfer = {}


for k, v in old_state.items():

    if (
        k
        in new_backbone_state
        and new_backbone_state[
            k
        ].shape
        == v.shape
    ):
        transfer[
            k
        ] = v


dual_model.backbone.load_state_dict(
    transfer,
    strict=False,
)


print(
    "Transferred backbone tensors:",
    len(
        transfer
    ),
    "/",
    len(
        new_backbone_state
    ),
)

print("Classes:", CLASS_TO_IDX)
print("STATUS: DUAL_VIEW_MODEL_READY")


In [ ]:
# CELL 12 — TRAIN DUAL-VIEW CLASSIFIER (UP TO 150 EPOCHS)

DUAL_EPOCHS = 150
DUAL_LR = 1e-5
PATIENCE = 30


BEST_DUAL = (
    CLS_ROOT
    / "best_dualview_effnetv2s_macro_f1.pth"
)

LAST_DUAL = (
    CLS_ROOT
    / "last_dualview_effnetv2s.pth"
)

HISTORY_CSV = (
    CLS_ROOT
    / "dualview_training_history.csv"
)


criterion = nn.CrossEntropyLoss(
    weight=torch.tensor(
        class_weights_np,
        dtype=torch.float32,
        device=DEVICE,
    ),
    label_smoothing=0.03,
)


optimizer = torch.optim.AdamW(
    dual_model.parameters(),
    lr=DUAL_LR,
    weight_decay=1e-4,
)


scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer,
    T_max=DUAL_EPOCHS,
    eta_min=2e-7,
)


scaler = torch.amp.GradScaler(
    "cuda"
)


def evaluate_dual(
    model,
    loader,
    return_embeddings=False,
):
    model.eval()

    yt = []
    yp = []
    probs_all = []
    emb_all = []

    with torch.no_grad():

        for context, masked, y in loader:

            context = context.to(
                DEVICE,
                non_blocking=True,
            )

            masked = masked.to(
                DEVICE,
                non_blocking=True,
            )

            y = torch.as_tensor(
                y,
                device=DEVICE,
            )


            with torch.autocast(
                "cuda",
                dtype=torch.float16,
            ):

                logits, emb = model(
                    context,
                    masked,
                    return_embedding=True,
                )


            probs = torch.softmax(
                logits.float(),
                dim=1,
            )

            pred = probs.argmax(
                dim=1
            )


            yt.extend(
                y.cpu().tolist()
            )

            yp.extend(
                pred.cpu().tolist()
            )

            probs_all.append(
                probs.cpu().numpy()
            )


            if return_embeddings:
                emb_all.append(
                    emb.float().cpu().numpy()
                )


    yt = np.asarray(
        yt
    )

    yp = np.asarray(
        yp
    )

    probs_all = np.concatenate(
        probs_all,
        axis=0,
    )


    metrics = {
        "accuracy": accuracy_score(
            yt,
            yp,
        ),
        "macro_precision": precision_score(
            yt,
            yp,
            average="macro",
            zero_division=0,
        ),
        "macro_recall": recall_score(
            yt,
            yp,
            average="macro",
            zero_division=0,
        ),
        "macro_f1": f1_score(
            yt,
            yp,
            average="macro",
            zero_division=0,
        ),
        "weighted_f1": f1_score(
            yt,
            yp,
            average="weighted",
            zero_division=0,
        ),
    }


    if return_embeddings:
        emb_all = np.concatenate(
            emb_all,
            axis=0,
        )

        return (
            metrics,
            yt,
            yp,
            probs_all,
            emb_all,
        )


    return (
        metrics,
        yt,
        yp,
        probs_all,
    )


history = []

best_f1 = -1.0
best_epoch = -1
no_improve = 0


for epoch in range(
    1,
    DUAL_EPOCHS + 1,
):

    dual_model.train()

    total_loss = 0.0
    seen = 0


    for context, masked, y in train_loader:

        context = context.to(
            DEVICE,
            non_blocking=True,
        )

        masked = masked.to(
            DEVICE,
            non_blocking=True,
        )

        y = torch.as_tensor(
            y,
            device=DEVICE,
        )


        optimizer.zero_grad(
            set_to_none=True
        )


        with torch.autocast(
            "cuda",
            dtype=torch.float16,
        ):

            logits = dual_model(
                context,
                masked,
            )

            loss = criterion(
                logits,
                y,
            )


        scaler.scale(
            loss
        ).backward()

        scaler.unscale_(
            optimizer
        )

        torch.nn.utils.clip_grad_norm_(
            dual_model.parameters(),
            5.0,
        )

        scaler.step(
            optimizer
        )

        scaler.update()


        total_loss += (
            float(
                loss.detach().cpu()
            )
            * context.size(
                0
            )
        )

        seen += context.size(
            0
        )


    scheduler.step()


    val_metrics, _, _, _ = evaluate_dual(
        dual_model,
        val_loader,
    )


    row = {
        "epoch": epoch,
        "train_loss": total_loss
        / max(
            seen,
            1,
        ),
        "lr": optimizer.param_groups[
            0
        ][
            "lr"
        ],
        **{
            f"val_{k}": float(
                v
            )
            for k, v
            in val_metrics.items()
        },
    }


    history.append(
        row
    )

    pd.DataFrame(
        history
    ).to_csv(
        HISTORY_CSV,
        index=False,
    )


    payload = {
        "model_state_dict": dual_model.state_dict(),
        "model_name": MODEL_NAME,
        "class_to_idx": CLASS_TO_IDX,
        "known_classes": KNOWN_CLASSES,
        "input_size": CLS_SIZE,
        "context_padding": CONTEXT_PAD,
        "mean": MEAN,
        "std": STD,
        "epoch": epoch,
        "val_metrics": {
            k: float(
                v
            )
            for k, v
            in val_metrics.items()
        },
        "architecture": "DualViewEfficientNetV2S",
    }


    torch.save(
        payload,
        LAST_DUAL,
    )


    if (
        val_metrics[
            "macro_f1"
        ]
        > best_f1
    ):

        best_f1 = float(
            val_metrics[
                "macro_f1"
            ]
        )

        best_epoch = epoch

        no_improve = 0

        torch.save(
            payload,
            BEST_DUAL,
        )

    else:
        no_improve += 1


    print(
        f"Epoch {epoch:03d}/{DUAL_EPOCHS}"
        f" | loss={row['train_loss']:.4f}"
        f" | Acc={val_metrics['accuracy']:.4f}"
        f" | MacroF1={val_metrics['macro_f1']:.4f}"
        f" | Best={best_f1:.4f}"
    )


    if no_improve >= PATIENCE:
        print(
            f"Early stopping after {PATIENCE} epochs without improvement."
        )
        break


print("Best epoch   :", best_epoch)
print("Best Macro-F1:", best_f1)
print("Best model   :", BEST_DUAL)
print("STATUS: DUAL_VIEW_TRAINED")


In [ ]:
# CELL 13 — OOD PROTOTYPES + FINAL CLASSIFIER CALIBRATION

best_ckpt = torch.load(
    BEST_DUAL,
    map_location=DEVICE,
    weights_only=False,
)

dual_model.load_state_dict(
    best_ckpt[
        "model_state_dict"
    ]
)

dual_model.eval()


(
    train_metrics,
    train_true,
    train_pred,
    train_probs,
    train_emb,
) = evaluate_dual(
    dual_model,
    train_eval_loader,
    return_embeddings=True,
)


(
    val_metrics,
    val_true,
    val_pred,
    val_probs,
    val_emb,
) = evaluate_dual(
    dual_model,
    val_loader,
    return_embeddings=True,
)


train_emb = train_emb / np.maximum(
    np.linalg.norm(
        train_emb,
        axis=1,
        keepdims=True,
    ),
    1e-9,
)

val_emb = val_emb / np.maximum(
    np.linalg.norm(
        val_emb,
        axis=1,
        keepdims=True,
    ),
    1e-9,
)


PROTOTYPES = {}
SIM_THRESHOLDS = {}


for class_name in KNOWN_CLASSES:

    ci = CLASS_TO_IDX[
        class_name
    ]

    idx_train = np.where(
        train_true
        == ci
    )[
        0
    ]

    if len(
        idx_train
    ) == 0:
        continue


    proto = train_emb[
        idx_train
    ].mean(
        axis=0
    )

    proto /= max(
        np.linalg.norm(
            proto
        ),
        1e-9,
    )

    PROTOTYPES[
        class_name
    ] = proto.astype(
        np.float32
    )


    idx_val = np.where(
        val_true
        == ci
    )[
        0
    ]

    if len(
        idx_val
    ) > 0:

        sims = val_emb[
            idx_val
        ] @ proto

        SIM_THRESHOLDS[
            class_name
        ] = float(
            np.percentile(
                sims,
                3.0,
            )
        )


PROTOTYPE_NPZ = (
    CAL_ROOT
    / "ood_prototypes.npz"
)

np.savez(
    PROTOTYPE_NPZ,
    **{
        f"proto__{k}": v
        for k, v
        in PROTOTYPES.items()
    },
)


CALIBRATION_JSON = (
    CAL_ROOT
    / "ood_calibration.json"
)


calibration = {
    "known_classes": KNOWN_CLASSES,
    "class_to_idx": CLASS_TO_IDX,
    "similarity_thresholds": SIM_THRESHOLDS,
    "known_conf_threshold": 0.50,
    "known_margin_threshold": 0.08,
    "non_coral_threshold": 0.45,
    "note": (
        "Other/Unknown Coral is a pragmatic open-set rejection. "
        "There is no explicit unseen-species validation set."
    ),
}


CALIBRATION_JSON.write_text(
    json.dumps(
        calibration,
        indent=2,
    ),
    encoding="utf-8",
)


report = pd.DataFrame(
    classification_report(
        val_true,
        val_pred,
        target_names=[
            IDX_TO_CLASS[
                i
            ]
            for i
            in range(
                NCLS
            )
        ],
        zero_division=0,
        output_dict=True,
    )
).T


report.to_csv(
    CLS_ROOT
    / "final_validation_classification_report.csv"
)


np.savetxt(
    CLS_ROOT
    / "final_validation_confusion_matrix.csv",
    confusion_matrix(
        val_true,
        val_pred,
    ),
    fmt="%d",
    delimiter=",",
)


print("=" * 110)
print("FINAL CLASSIFIER VALIDATION")
print("=" * 110)

for k, v in val_metrics.items():
    print(
        f"{k:18s}: {v:.6f}"
    )

print("Calibration:", CALIBRATION_JSON)
print("STATUS: OOD_READY")


In [ ]:
# CELL 14 — FINAL INFERENCE:
# YOLO26m-SEG STUDENT PRIMARY + CORALSCOP FALLBACK

seg_student = YOLO(
    str(
        PROMOTED_SEG
    )
)


cal = json.loads(
    CALIBRATION_JSON.read_text(
        encoding="utf-8"
    )
)


proto_file = np.load(
    PROTOTYPE_NPZ
)


PROTOTYPES = {
    k.replace(
        "proto__",
        "",
    ): proto_file[
        k
    ]
    for k
    in proto_file.files
}


MASK_DUP_IOU = 0.65

REFINE_CONF_THRESHOLD = max(
    0.10,
    BEST_SEG_CONF
    * 0.80,
)

REFINE_MIN_MASK_BOX_RATIO = 0.10
REFINE_MAX_MASK_BOX_RATIO = 1.60
REFINE_HIGH_OVERLAP = 0.70

MAX_DET = 500


def polygon_to_mask(
    polygon,
    H,
    W,
):
    mask = np.zeros(
        (
            H,
            W,
        ),
        dtype=np.uint8,
    )

    if polygon is None:
        return mask.astype(
            bool
        )

    pts = np.asarray(
        polygon,
        dtype=np.int32,
    ).reshape(
        -1,
        1,
        2,
    )

    if len(
        pts
    ) >= 3:
        cv2.fillPoly(
            mask,
            [
                pts
            ],
            1,
        )

    return mask.astype(
        bool
    )


def mask_iou(
    a,
    b,
):
    inter = np.logical_and(
        a,
        b,
    ).sum()

    union = np.logical_or(
        a,
        b,
    ).sum()

    return float(
        inter
        / max(
            union,
            1,
        )
    )


def student_instances(
    rgb,
):
    H, W = rgb.shape[
        :2
    ]

    result = seg_student.predict(
        source=rgb,
        imgsz=SEG_IMGSZ,
        conf=BEST_SEG_CONF,
        iou=0.70,
        max_det=MAX_DET,
        device=0,
        verbose=False,
    )[
        0
    ]


    if (
        result.boxes
        is None
        or len(
            result.boxes
        ) == 0
        or result.masks
        is None
    ):
        return []


    boxes = (
        result.boxes.xyxy
        .detach()
        .cpu()
        .numpy()
        .astype(
            np.float32
        )
    )

    confs = (
        result.boxes.conf
        .detach()
        .cpu()
        .numpy()
        .astype(
            np.float32
        )
    )


    polygons = result.masks.xy


    instances = []


    for box, conf, poly in zip(
        boxes,
        confs,
        polygons,
    ):

        mask = polygon_to_mask(
            poly,
            H,
            W,
        )

        if (
            mask.sum()
            < MASK_MIN_PIXELS
        ):
            continue


        mbox = mask_bbox(
            mask
        )

        if mbox is None:
            continue


        ratio = float(
            mask.sum()
        ) / max(
            box_area(
                box
            ),
            1.0,
        )


        instances.append(
            {
                "box": box.astype(
                    np.float32
                ),
                "mask": mask,
                "conf": float(
                    conf
                ),
                "mask_box_ratio": ratio,
                "refined": False,
            }
        )


    return instances


def needs_coralscop_refinement(
    instance,
    all_instances,
):
    if (
        instance[
            "conf"
        ]
        < REFINE_CONF_THRESHOLD
    ):
        return True

    ratio = float(
        instance[
            "mask_box_ratio"
        ]
    )

    if (
        ratio
        < REFINE_MIN_MASK_BOX_RATIO
        or ratio
        > REFINE_MAX_MASK_BOX_RATIO
    ):
        return True


    for other in all_instances:

        if other is instance:
            continue

        if (
            mask_iou(
                instance[
                    "mask"
                ],
                other[
                    "mask"
                ],
            )
            >= REFINE_HIGH_OVERLAP
        ):
            return True


    return False


def refine_instances(
    rgb,
    instances,
):
    if not instances:
        return []


    need_any = any(
        needs_coralscop_refinement(
            inst,
            instances,
        )
        for inst
        in instances
    )


    if need_any:
        coralscop_predictor.set_image(
            rgb
        )


    refined = []


    for inst in instances:

        if not needs_coralscop_refinement(
            inst,
            instances,
        ):
            refined.append(
                inst
            )
            continue


        with torch.inference_mode():

            masks, scores, _ = coralscop_predictor.predict(
                box=inst[
                    "box"
                ],
                multimask_output=True,
            )


        idx = int(
            np.argmax(
                scores
            )
        )

        mask = masks[
            idx
        ].astype(
            bool
        )


        if (
            mask.sum()
            >= MASK_MIN_PIXELS
        ):

            mb = mask_bbox(
                mask
            )

            if mb is not None:

                inst = {
                    **inst,
                    "mask": mask,
                    "box": mb,
                    "teacher_mask_score": float(
                        scores[
                            idx
                        ]
                    ),
                    "refined": True,
                }


        refined.append(
            inst
        )


    # Mask-IoU duplicate suppression.
    refined.sort(
        key=lambda x: x[
            "conf"
        ],
        reverse=True,
    )


    kept = []


    for inst in refined:

        duplicate = any(
            mask_iou(
                inst[
                    "mask"
                ],
                k[
                    "mask"
                ],
            )
            >= MASK_DUP_IOU
            for k
            in kept
        )

        if not duplicate:
            kept.append(
                inst
            )


    return kept


def make_dual_views(
    rgb,
    mask,
    box,
):
    H, W = rgb.shape[
        :2
    ]

    x1, y1, x2, y2 = padded_box(
        box,
        W,
        H,
        CONTEXT_PAD,
    )

    context = rgb[
        y1:y2,
        x1:x2,
    ].copy()

    local_mask = mask[
        y1:y2,
        x1:x2,
    ]

    masked = context.copy()

    masked[
        ~local_mask
    ] = 0

    return (
        Image.fromarray(
            context
        ),
        Image.fromarray(
            masked
        ),
    )


def classify_instances(
    rgb,
    instances,
):
    if not instances:
        return instances


    contexts = []
    maskeds = []


    for inst in instances:

        context, masked = make_dual_views(
            rgb,
            inst[
                "mask"
            ],
            inst[
                "box"
            ],
        )

        contexts.append(
            val_tf(
                context
            )
        )

        maskeds.append(
            val_tf(
                masked
            )
        )


    context_t = torch.stack(
        contexts
    ).to(
        DEVICE
    )

    masked_t = torch.stack(
        maskeds
    ).to(
        DEVICE
    )


    with torch.inference_mode():
        with torch.autocast(
            "cuda",
            dtype=torch.float16,
        ):
            logits, emb = dual_model(
                context_t,
                masked_t,
                return_embedding=True,
            )


    probs = torch.softmax(
        logits.float(),
        dim=1,
    ).cpu().numpy()


    emb = emb.float().cpu().numpy()

    emb /= np.maximum(
        np.linalg.norm(
            emb,
            axis=1,
            keepdims=True,
        ),
        1e-9,
    )


    for inst, p, e in zip(
        instances,
        probs,
        emb,
    ):

        order = np.argsort(
            p
        )[
            ::-1
        ]

        top = int(
            order[
                0
            ]
        )

        second = int(
            order[
                1
            ]
        )

        raw = IDX_TO_CLASS[
            top
        ]

        conf = float(
            p[
                top
            ]
        )

        margin = float(
            p[
                top
            ]
            - p[
                second
            ]
        )

        non_prob = float(
            p[
                CLASS_TO_IDX[
                    "NON_CORAL"
                ]
            ]
        )


        if (
            raw
            == "NON_CORAL"
            or non_prob
            >= float(
                cal[
                    "non_coral_threshold"
                ]
            )
        ):

            label = "Non-Coral"
            similarity = None

        else:

            proto = PROTOTYPES.get(
                raw
            )

            similarity = (
                float(
                    e
                    @ proto
                )
                if proto
                is not None
                else -1.0
            )

            if (
                conf
                < float(
                    cal[
                        "known_conf_threshold"
                    ]
                )
                or margin
                < float(
                    cal[
                        "known_margin_threshold"
                    ]
                )
                or similarity
                < float(
                    cal[
                        "similarity_thresholds"
                    ].get(
                        raw,
                        -1.0,
                    )
                )
            ):

                label = "Other/Unknown Coral"

            else:
                label = raw


        inst[
            "label"
        ] = label

        inst[
            "cls_conf"
        ] = conf

        inst[
            "cls_margin"
        ] = margin

        inst[
            "prob_vector"
        ] = p.astype(
            np.float32
        )

        inst[
            "similarity"
        ] = similarity


    return instances


def final_frame_instances(
    rgb,
):
    instances = student_instances(
        rgb
    )

    instances = refine_instances(
        rgb,
        instances,
    )

    instances = classify_instances(
        rgb,
        instances,
    )

    return instances


print("Primary model :", PROMOTED_SEG)
print("Primary conf  :", BEST_SEG_CONF)
print("Fallback      : CoralSCOP only for difficult masks")
print("Classifier    :", BEST_DUAL)
print("STATUS: FINAL_INFERENCE_READY")


In [ ]:
# CELL 15 — SAM2.1 VIDEO MEMORY CONFIG

from sam2.sam2_video_predictor import SAM2VideoPredictor


SAM2_MODEL_ID = "facebook/sam2-hiera-large"

KEYFRAME_STRIDE = 5

MAX_SEED_GAP = 15

SEED_MASK_IOU = 0.20

SEED_BOX_IOU = 0.12

MAX_SAM2_TRACKS = 60

CLS_STRIDE = 3

EMA_ALPHA = 0.25


def centroid(
    b,
):
    x1, y1, x2, y2 = map(
        float,
        b,
    )

    return np.array(
        [
            (
                x1
                + x2
            )
            / 2.0,
            (
                y1
                + y2
            )
            / 2.0,
        ],
        dtype=np.float32,
    )


def associate_seed_tracks(
    tracks,
    dets,
    fi,
    W,
    H,
):
    active = [
        tid
        for tid, t
        in tracks.items()
        if (
            fi
            - t[
                "last_frame"
            ]
        )
        <= MAX_SEED_GAP
    ]


    if (
        not active
        or not dets
    ):
        return (
            [],
            list(
                range(
                    len(
                        dets
                    )
                )
            ),
        )


    cost = np.ones(
        (
            len(
                active
            ),
            len(
                dets
            ),
        ),
        dtype=np.float32,
    )


    diag = max(
        math.hypot(
            W,
            H,
        ),
        1.0,
    )


    for i, tid in enumerate(
        active
    ):

        for j, d in enumerate(
            dets
        ):

            mi = mask_iou(
                tracks[
                    tid
                ][
                    "last_mask"
                ],
                d[
                    "mask"
                ],
            )

            bi = box_iou(
                tracks[
                    tid
                ][
                    "last_box"
                ],
                d[
                    "box"
                ],
            )

            cd = (
                np.linalg.norm(
                    centroid(
                        tracks[
                            tid
                        ][
                            "last_box"
                        ]
                    )
                    - centroid(
                        d[
                            "box"
                        ]
                    )
                )
                / diag
            )


            cost[
                i,
                j
            ] = (
                1.0
                - max(
                    mi,
                    bi,
                    max(
                        0.0,
                        1.0
                        - 4.0
                        * cd,
                    ),
                )
            )


    rr, cc = linear_sum_assignment(
        cost
    )


    matches = []

    used = set()


    for r, c in zip(
        rr,
        cc,
    ):

        tid = active[
            r
        ]

        d = dets[
            c
        ]


        if (
            mask_iou(
                tracks[
                    tid
                ][
                    "last_mask"
                ],
                d[
                    "mask"
                ],
            )
            < SEED_MASK_IOU
            and box_iou(
                tracks[
                    tid
                ][
                    "last_box"
                ],
                d[
                    "box"
                ],
            )
            < SEED_BOX_IOU
        ):
            continue


        matches.append(
            (
                tid,
                c,
            )
        )

        used.add(
            c
        )


    unmatched = [
        j
        for j
        in range(
            len(
                dets
            )
        )
        if j
        not in used
    ]


    return (
        matches,
        unmatched,
    )


VIDEO_ZIP = (
    WORKSPACE
    / "data"
    / "Coral_video"
    / "coral_video.zip"
)


if not VIDEO_ZIP.exists():
    raise FileNotFoundError(
        VIDEO_ZIP
    )


print("SAM2 model:", SAM2_MODEL_ID)
print("Video ZIP :", VIDEO_ZIP)
print("STATUS: VIDEO_CONFIG_READY")


In [ ]:
# CELL 16 — FINAL VIDEO:
# YOLO26m-SEG STUDENT → CONDITIONAL CORALSCOP → SAM2 → DUAL-VIEW LABELS

VIDEO_WORK = (
    CACHE_ROOT
    / "CoralGuardX_V2_Video"
)


if VIDEO_WORK.exists():
    shutil.rmtree(
        VIDEO_WORK
    )


(
    VIDEO_WORK
    / "input"
).mkdir(
    parents=True,
    exist_ok=True,
)


with zipfile.ZipFile(
    VIDEO_ZIP,
    "r",
) as z:
    z.extractall(
        VIDEO_WORK
        / "input"
    )


exts = {
    ".mp4",
    ".mov",
    ".avi",
    ".mkv",
    ".m4v",
    ".webm",
    ".mpg",
    ".mpeg",
}


video_paths = sorted(
    [
        p
        for p
        in (
            VIDEO_WORK
            / "input"
        ).rglob(
            "*"
        )
        if p.is_file()
        and p.suffix.lower()
        in exts
    ]
)


if not video_paths:
    raise RuntimeError(
        "No videos found."
    )


def extract_frames(
    video_path,
    frame_dir,
):

    if frame_dir.exists():
        shutil.rmtree(
            frame_dir
        )

    frame_dir.mkdir(
        parents=True,
        exist_ok=True,
    )


    cap = cv2.VideoCapture(
        str(
            video_path
        )
    )


    if not cap.isOpened():
        raise RuntimeError(
            f"Could not open {video_path}"
        )


    fps = float(
        cap.get(
            cv2.CAP_PROP_FPS
        )
    )


    if (
        not np.isfinite(
            fps
        )
        or fps <= 0
    ):
        fps = 25.0


    W = int(
        cap.get(
            cv2.CAP_PROP_FRAME_WIDTH
        )
    )

    H = int(
        cap.get(
            cv2.CAP_PROP_FRAME_HEIGHT
        )
    )


    n = 0


    while True:

        ok, frame = cap.read()

        if not ok:
            break


        cv2.imwrite(
            str(
                frame_dir
                / f"{n:06d}.jpg"
            ),
            frame,
            [
                int(
                    cv2.IMWRITE_JPEG_QUALITY
                ),
                95,
            ],
        )

        n += 1


    cap.release()


    return (
        n,
        fps,
        W,
        H,
    )


video_summaries = []


for vi, vp in enumerate(
    video_paths,
    start=1,
):

    stem = vp.stem.replace(
        " ",
        "_",
    )


    out_dir = (
        VIDEO_ROOT
        / stem
    )

    out_dir.mkdir(
        parents=True,
        exist_ok=True,
    )


    final_video = (
        out_dir
        / f"{stem}_CoralGuardX_V2_FINAL.mp4"
    )

    final_json = (
        out_dir
        / f"{stem}_summary.json"
    )


    if (
        final_video.exists()
        and final_json.exists()
    ):

        print(
            "SKIP:",
            vp.name,
        )

        video_summaries.append(
            json.loads(
                final_json.read_text(
                    encoding="utf-8"
                )
            )
        )

        continue


    print(
        "\n"
        + "=" * 115
    )

    print(
        f"VIDEO {vi}/{len(video_paths)}: {vp.name}"
    )


    frame_dir = (
        VIDEO_WORK
        / stem
        / "frames"
    )


    n_frames, fps, W, H = extract_frames(
        vp,
        frame_dir,
    )


    # --------------------------------------------------------
    # PASS A:
    # Student segmentation on sparse keyframes.
    # CoralSCOP only refines difficult student masks.
    # --------------------------------------------------------

    tracks = {}

    next_id = 1

    refined_count = 0

    student_instance_count = 0


    for fi in range(
        0,
        n_frames,
        KEYFRAME_STRIDE,
    ):

        bgr = cv2.imread(
            str(
                frame_dir
                / f"{fi:06d}.jpg"
            )
        )

        rgb = cv2.cvtColor(
            bgr,
            cv2.COLOR_BGR2RGB,
        )


        dets = final_frame_instances(
            rgb
        )


        student_instance_count += len(
            dets
        )

        refined_count += sum(
            int(
                d.get(
                    "refined",
                    False,
                )
            )
            for d
            in dets
        )


        matches, unmatched = associate_seed_tracks(
            tracks,
            dets,
            fi,
            W,
            H,
        )


        for tid, di in matches:

            d = dets[
                di
            ]

            t = tracks[
                tid
            ]

            t[
                "last_frame"
            ] = fi

            t[
                "last_box"
            ] = d[
                "box"
            ]

            t[
                "last_mask"
            ] = d[
                "mask"
            ]

            t[
                "hits"
            ] += 1

            t[
                "score_sum"
            ] += d[
                "conf"
            ]


        for di in unmatched:

            d = dets[
                di
            ]

            tid = next_id

            next_id += 1

            tracks[
                tid
            ] = {
                "id": tid,
                "seed_frame": fi,
                "seed_box": d[
                    "box"
                ].copy(),
                "seed_mask": d[
                    "mask"
                ].copy(),
                "last_frame": fi,
                "last_box": d[
                    "box"
                ].copy(),
                "last_mask": d[
                    "mask"
                ].copy(),
                "hits": 1,
                "score_sum": d[
                    "conf"
                ],
            }


        if (
            fi
            % (
                KEYFRAME_STRIDE
                * 20
            )
            == 0
        ):

            print(
                f"Seed pass {fi}/{n_frames}"
                f" | tracks={len(tracks)}"
                f" | refined={refined_count}"
            )


    track_list = list(
        tracks.values()
    )


    track_list.sort(
        key=lambda t: (
            t[
                "hits"
            ],
            t[
                "score_sum"
            ]
            / max(
                t[
                    "hits"
                ],
                1,
            ),
        ),
        reverse=True,
    )


    track_list = track_list[
        :MAX_SAM2_TRACKS
    ]


    if not track_list:
        raise RuntimeError(
            f"No coral seeds found in {vp.name}"
        )


    print(
        "SAM2 seed tracks:",
        len(
            track_list
        )
    )


    # --------------------------------------------------------
    # Free student + CoralSCOP GPU before SAM2 large memory.
    # --------------------------------------------------------

    try:
        coralscop_predictor.reset_image()
    except Exception:
        pass


    coralscop_model.to(
        "cpu"
    )


    del seg_student

    gc.collect()

    torch.cuda.empty_cache()


    sam2_predictor = SAM2VideoPredictor.from_pretrained(
        SAM2_MODEL_ID
    )


    state = sam2_predictor.init_state(
        video_path=str(
            frame_dir
        ),
        offload_video_to_cpu=True,
        offload_state_to_cpu=False,
        async_loading_frames=False,
    )


    # Seed SAM2 with the actual student/teacher-refined mask,
    # not merely the box.
    for t in track_list:

        sam2_predictor.add_new_mask(
            inference_state=state,
            frame_idx=int(
                t[
                    "seed_frame"
                ]
            ),
            obj_id=int(
                t[
                    "id"
                ]
            ),
            mask=t[
                "seed_mask"
            ],
        )


    ffmpeg = shutil.which(
        "ffmpeg"
    )


    if ffmpeg is None:
        raise RuntimeError(
            "FFmpeg is not on PATH."
        )


    cmd = [
        ffmpeg,
        "-loglevel",
        "error",
        "-y",
        "-f",
        "rawvideo",
        "-vcodec",
        "rawvideo",
        "-pix_fmt",
        "bgr24",
        "-s",
        f"{W}x{H}",
        "-r",
        f"{fps:.8f}",
        "-i",
        "pipe:0",
        "-i",
        str(
            vp
        ),
        "-map",
        "0:v:0",
        "-map",
        "1:a:0?",
        "-c:v",
        "libx264",
        "-preset",
        "slow",
        "-crf",
        "12",
        "-pix_fmt",
        "yuv420p",
        "-c:a",
        "aac",
        "-b:a",
        "192k",
        "-movflags",
        "+faststart",
        "-shortest",
        str(
            final_video
        ),
    ]


    ff = subprocess.Popen(
        cmd,
        stdin=subprocess.PIPE,
        stdout=subprocess.DEVNULL,
        stderr=subprocess.PIPE,
    )


    cls_state = {}

    transitions = 0

    switches = 0

    frames_done = 0

    propagated_masks = 0


    with torch.inference_mode():

        for (
            out_fi,
            out_ids,
            out_logits,
        ) in sam2_predictor.propagate_in_video(
            state,
            start_frame_idx=0,
        ):

            fi = int(
                out_fi
            )

            bgr = cv2.imread(
                str(
                    frame_dir
                    / f"{fi:06d}.jpg"
                )
            )

            rgb = cv2.cvtColor(
                bgr,
                cv2.COLOR_BGR2RGB,
            )


            masks = []

            boxes = []

            ids = []


            for oi, obj_id in enumerate(
                out_ids
            ):

                lg = out_logits[
                    oi
                ]

                if (
                    lg.ndim
                    == 3
                ):
                    lg = lg[
                        0
                    ]


                mask = (
                    lg
                    > 0.0
                ).detach().cpu().numpy().astype(
                    bool
                )


                if (
                    mask.sum()
                    < MASK_MIN_PIXELS
                ):
                    continue


                box = mask_bbox(
                    mask
                )


                if box is None:
                    continue


                masks.append(
                    mask
                )

                boxes.append(
                    box
                )

                ids.append(
                    int(
                        obj_id
                    )
                )


            propagated_masks += len(
                masks
            )


            if (
                masks
                and fi
                % CLS_STRIDE
                == 0
            ):

                temp_instances = [
                    {
                        "mask": m,
                        "box": b,
                    }
                    for m, b
                    in zip(
                        masks,
                        boxes,
                    )
                ]


                temp_instances = classify_instances(
                    rgb,
                    temp_instances,
                )


                for obj_id, inst in zip(
                    ids,
                    temp_instances,
                ):

                    p = inst[
                        "prob_vector"
                    ]

                    state_cls = cls_state.get(
                        obj_id
                    )


                    if state_cls is None:

                        smooth = p.copy()

                        state_cls = {
                            "prob": smooth,
                            "last_label": None,
                        }

                    else:

                        smooth = (
                            EMA_ALPHA
                            * p
                            + (
                                1.0
                                - EMA_ALPHA
                            )
                            * state_cls[
                                "prob"
                            ]
                        )

                        smooth /= max(
                            float(
                                smooth.sum()
                            ),
                            1e-9,
                        )

                        state_cls[
                            "prob"
                        ] = smooth


                    order = np.argsort(
                        state_cls[
                            "prob"
                        ]
                    )[
                        ::-1
                    ]

                    top = int(
                        order[
                            0
                        ]
                    )

                    second = int(
                        order[
                            1
                        ]
                    )

                    raw = IDX_TO_CLASS[
                        top
                    ]

                    top_conf = float(
                        state_cls[
                            "prob"
                        ][
                            top
                        ]
                    )

                    margin = float(
                        state_cls[
                            "prob"
                        ][
                            top
                        ]
                        - state_cls[
                            "prob"
                        ][
                            second
                        ]
                    )

                    non_prob = float(
                        state_cls[
                            "prob"
                        ][
                            CLASS_TO_IDX[
                                "NON_CORAL"
                            ]
                        ]
                    )


                    if (
                        raw
                        == "NON_CORAL"
                        or non_prob
                        >= float(
                            cal[
                                "non_coral_threshold"
                            ]
                        )
                    ):

                        label = "Non-Coral"

                    elif (
                        top_conf
                        < float(
                            cal[
                                "known_conf_threshold"
                            ]
                        )
                        or margin
                        < float(
                            cal[
                                "known_margin_threshold"
                            ]
                        )
                        or inst[
                            "label"
                        ]
                        == "Other/Unknown Coral"
                    ):

                        label = "Other/Unknown Coral"

                    else:
                        label = raw


                    if (
                        state_cls[
                            "last_label"
                        ]
                        is not None
                    ):

                        transitions += 1

                        if (
                            label
                            != state_cls[
                                "last_label"
                            ]
                        ):
                            switches += 1


                    state_cls[
                        "last_label"
                    ] = label

                    state_cls[
                        "label"
                    ] = label

                    state_cls[
                        "confidence"
                    ] = top_conf

                    cls_state[
                        obj_id
                    ] = state_cls


            overlay = bgr.copy()


            for obj_id, mask, box in zip(
                ids,
                masks,
                boxes,
            ):

                s = cls_state.get(
                    obj_id,
                    {
                        "label": "Analyzing Coral",
                        "confidence": 0.0,
                    },
                )

                label = s[
                    "label"
                ]

                conf = float(
                    s.get(
                        "confidence",
                        0.0,
                    )
                )


                if label == "Non-Coral":
                    color = (
                        120,
                        120,
                        120,
                    )

                elif label == "Other/Unknown Coral":
                    color = (
                        0,
                        165,
                        255,
                    )

                else:
                    color = (
                        0,
                        255,
                        0,
                    )


                overlay[
                    mask
                ] = (
                    0.55
                    * overlay[
                        mask
                    ]
                    + 0.45
                    * np.asarray(
                        color,
                        dtype=np.float32,
                    )
                ).astype(
                    np.uint8
                )


                contours, _ = cv2.findContours(
                    mask.astype(
                        np.uint8
                    ),
                    cv2.RETR_EXTERNAL,
                    cv2.CHAIN_APPROX_SIMPLE,
                )


                cv2.drawContours(
                    bgr,
                    contours,
                    -1,
                    color,
                    3,
                )


                x1, y1, x2, y2 = [
                    int(
                        round(
                            z
                        )
                    )
                    for z
                    in box
                ]


                text = (
                    f"ID {obj_id} | {label} | {conf:.2f}"
                )


                cv2.putText(
                    bgr,
                    text,
                    (
                        x1,
                        max(
                            25,
                            y1 - 8,
                        ),
                    ),
                    cv2.FONT_HERSHEY_SIMPLEX,
                    0.72,
                    color,
                    2,
                    cv2.LINE_AA,
                )


            bgr = cv2.addWeighted(
                overlay,
                0.45,
                bgr,
                0.55,
                0,
            )


            ff.stdin.write(
                bgr.tobytes()
            )


            frames_done += 1


            if (
                frames_done
                % 100
                == 0
            ):
                print(
                    f"Rendered {frames_done}/{n_frames}"
                    f" | propagated masks={propagated_masks}"
                )


    ff.stdin.close()

    ff.wait()


    ff_err = ff.stderr.read().decode(
        "utf-8",
        errors="ignore",
    )


    if ff.returncode != 0:
        raise RuntimeError(
            ff_err[
                -5000:
            ]
        )


    summary = {
        "video": vp.name,
        "frames": int(
            n_frames
        ),
        "student_keyframe_instances": int(
            student_instance_count
        ),
        "coralscop_fallback_refinements": int(
            refined_count
        ),
        "seed_tracks": int(
            len(
                track_list
            )
        ),
        "propagated_masks": int(
            propagated_masks
        ),
        "label_transitions": int(
            transitions
        ),
        "label_switches": int(
            switches
        ),
        "species_flicker_rate": (
            float(
                switches
                / transitions
            )
            if transitions
            else 0.0
        ),
        "seg_student": str(
            PROMOTED_SEG
        ),
        "student_conf": float(
            BEST_SEG_CONF
        ),
        "coralscop": str(
            CORALSCOP_CKPT
        ),
        "sam2": SAM2_MODEL_ID,
        "dual_view_classifier": str(
            BEST_DUAL
        ),
    }


    final_json.write_text(
        json.dumps(
            summary,
            indent=2,
            ensure_ascii=False,
        ),
        encoding="utf-8",
    )


    video_summaries.append(
        summary
    )


    print(
        json.dumps(
            summary,
            indent=2,
        )
    )

    print(
        "Saved:",
        final_video,
    )


    # Cleanup SAM2 and restore student/CoralSCOP for the next video.
    del sam2_predictor, state

    gc.collect()

    torch.cuda.empty_cache()


    coralscop_model.to(
        DEVICE
    )


    seg_student = YOLO(
        str(
            PROMOTED_SEG
        )
    )


print("STATUS: FINAL_VIDEO_PIPELINE_COMPLETE")


In [ ]:
# CELL 17 — FINAL PROJECT SUMMARY

summary_df = pd.DataFrame(
    video_summaries
)


VIDEO_SUMMARY_CSV = (
    VIDEO_ROOT
    / "CoralGuardX_V2_all_video_summary.csv"
)


summary_df.to_csv(
    VIDEO_SUMMARY_CSV,
    index=False,
)


FINAL_JSON = (
    FINAL_ROOT
    / "CoralGuardX_V2_FINAL_PROJECT_SUMMARY.json"
)


final_summary = {
    "architecture": (
        "GT Boxes -> CoralSCOP Teacher Pseudo Masks -> "
        "YOLO26m-SEG Student with compatible YOLO26m detector weight transfer -> "
        "Conditional CoralSCOP fallback -> "
        "Dual-View EfficientNetV2-S -> OOD gate -> "
        "SAM2.1 temporal mask memory"
    ),
    "detection_teacher": str(
        DETECTOR_TEACHER
    ),
    "mask_teacher": str(
        CORALSCOP_CKPT
    ),
    "segmentation_student": str(
        PROMOTED_SEG
    ),
    "student_operating_confidence": float(
        BEST_SEG_CONF
    ),
    "student_pseudo_mask_metrics": SEG_METRICS,
    "dual_view_classifier": str(
        BEST_DUAL
    ),
    "classifier_validation": {
        k: float(
            v
        )
        for k, v
        in val_metrics.items()
    },
    "ood_calibration": str(
        CALIBRATION_JSON
    ),
    "sam2_model": SAM2_MODEL_ID,
    "known_species": KNOWN_CLASSES,
    "extra_outputs": [
        "NON_CORAL",
        "Other/Unknown Coral",
    ],
    "videos_processed": int(
        len(
            summary_df
        )
    ),
    "held_out_detector_test_used": False,
    "segmentation_validation_note": (
        "Mask metrics are measured against CoralSCOP pseudo masks, "
        "not independent human instance-mask annotations."
    ),
    "label_limitation": (
        "Dead Coral and Algae are not explicit output classes because "
        "the current dataset does not provide those labels."
    ),
}


FINAL_JSON.write_text(
    json.dumps(
        final_summary,
        indent=2,
        ensure_ascii=False,
    ),
    encoding="utf-8",
)


print("=" * 120)
print("CORALGUARD-X V2 FINAL COMPLETE")
print("=" * 120)
print("Teacher detector :", DETECTOR_TEACHER)
print("Mask teacher     :", CORALSCOP_CKPT)
print("SEG student      :", PROMOTED_SEG)
print("Student conf     :", BEST_SEG_CONF)
print("Classifier       :", BEST_DUAL)
print("Videos           :", VIDEO_ROOT)
print("Summary          :", FINAL_JSON)
print("Held-out TEST    : NOT USED")
print("STATUS: COMPLETE")


## Final outputs

```text
D:\Coral_Local_Workspace\
├── models\
│   └── best_yolo26m_seg_student.pt
│
└── outputs\
    └── CoralGuard_X_V2_FINAL\
        ├── 01_Teacher_Masks\
        ├── 02_YOLO26m_SEG_Student\
        │   ├── teacher_mask_quality_audit.csv
        │   ├── student_confidence_sweep_original_GT.csv
        │   └── student_operating_config.json
        ├── 03_DualView_Data\
        ├── 04_DualView_Classifier\
        │   └── best_dualview_effnetv2s_macro_f1.pth
        ├── 05_OOD_Calibration\
        ├── 06_Final_Videos\
        └── CoralGuardX_V2_FINAL_PROJECT_SUMMARY.json
```

### Teacher → Student meaning in this notebook

This is **pseudo-label + transfer learning teacher/student**, not a fake claim of direct CoralSCOP logit distillation:

1. **CoralSCOP teacher** supplies instance-mask supervision.
2. **Existing coral YOLO26m detector** supplies compatible learned weights to the segmentation student.
3. **YOLO26m-seg student** learns to produce masks itself and becomes the primary deployment model.
4. CoralSCOP remains only as an expert fallback for difficult student masks.
